# Document Image Classification SD234504: EDA, Preprocessing, Modelling, dan Evaluasi

**Latar belakang.** Pengelompokan dokumen secara manual memerlukan waktu yang besar dan rentan terhadap kesalahan. Analisis ini bertujuan membangun model klasifikasi citra yang mampu menentukan jenis dokumen ke dalam 16 kategori secara otomatis.

**Tujuan.** Membangun model klasifikasi dengan akurasi setinggi mungkin pada data uji. Metrik evaluasi yang digunakan adalah *categorical accuracy*, yaitu proporsi prediksi yang sesuai dengan label sebenarnya.

**Struktur kode.** Notebook ini berfungsi sebagai alur analisis. Logika yang dapat digunakan ulang (pemuatan data, pembersihan, augmentasi, model, training, dan prediksi) berada di folder `src/`. Konfigurasi dan seluruh hyperparameter berada di `src/config.py`. Jalankan sel secara berurutan dari atas ke bawah.

## Tahap 1: Import Library dan Read Data
### 1.1 Import Library

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import torch
import torch.nn as nn
import torch.optim as optim

# Root proyek: folder di atas notebooks/ jika notebook dijalankan dari sana
cwd = os.getcwd()
PROJECT_ROOT = os.path.dirname(cwd) if os.path.basename(cwd) == "notebooks" else cwd
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import CONFIG, DATA_DIR, CACHE_DIR, CHECKPOINT_DIR, OUTPUT_DIR
from src.utils import setup_environment, get_device
from src.data import (
    load_competition_tables, read_image_metadata, remove_unreadable, add_md5,
    clean_duplicates, get_class_names, split_train_val, build_transforms,
    build_all_caches, build_dataloaders,
)
from src.model import build_model
from src.engine import train_model, load_checkpoint, predict_batches
from src.predict import write_submission

sns.set_theme(style="whitegrid")
setup_environment(CONFIG["random_state"])
DEVICE = get_device()
print("Device:", DEVICE)

### 1.2 Read file CSV
Membaca tiga file CSV kompetisi dari folder `data/`.
- `train.csv` berisi `id`, `filename`, `class_name`, dan `label`. Ini satu-satunya file yang memuat label.
- `test.csv` berisi `id` dan `filename`. Label test tidak diberikan dan tidak boleh dipakai dalam training.
- `sample_submission.csv` dipakai sebagai acuan format submission dan urutan id.

In [ ]:
train_df, test_df, sample_sub = load_competition_tables(DATA_DIR)
print("train:", train_df.shape, "| test:", test_df.shape, "| sample_submission:", sample_sub.shape)
display(train_df.head())
train_df.info()

In [ ]:
print("File train ditemukan:", train_df["path"].apply(os.path.isfile).sum(), "dari", len(train_df))
print("File test ditemukan :", test_df["path"].apply(os.path.isfile).sum(), "dari", len(test_df))

## Tahap 2: Exploratory Data Analysis (EDA)
### 2.1 Missing value dan duplikasi id
Memeriksa jumlah nilai kosong di setiap tabel, serta id dan filename yang duplikat. Duplikat di CSV berarti satu gambar tercatat lebih dari sekali, sehingga jumlah sampel dan distribusi kelas ikut terdistorsi.

In [ ]:
def missing_summary(df, name):
    # Jumlah dan persentase nilai kosong per kolom
    out = pd.DataFrame({"missing": df.isna().sum(), "persen": (df.isna().mean() * 100).round(3)})
    out.insert(0, "dataset", name)
    return out

raw_frames = {
    "train": train_df[["id", "filename", "class_name", "label"]],
    "test": test_df[["id", "filename"]],
    "sample_submission": sample_sub,
}
display(pd.concat([missing_summary(df, name) for name, df in raw_frames.items()]))

print("Duplikat id train  :", train_df["id"].duplicated().sum())
print("Duplikat filename  :", train_df["filename"].duplicated().sum())
print("Duplikat id test   :", test_df["id"].duplicated().sum())

### 2.2 Konsistensi label dan class_name
Memastikan setiap label memetakan ke satu `class_name`, label berada pada rentang 0 sampai 15, dan folder kelas pada path sesuai dengan `class_name`. Jika tidak konsisten, model belajar dari label yang salah.

In [ ]:
label_to_names = train_df.groupby("label")["class_name"].nunique()
name_to_labels = train_df.groupby("class_name")["label"].nunique()
print("Label unik                  :", train_df["label"].nunique())
print("Label di luar rentang 0-15  :", int((~train_df["label"].between(0, 15)).sum()))
print("Label dengan >1 class_name  :", label_to_names[label_to_names > 1].index.tolist())
print("class_name dengan >1 label  :", name_to_labels[name_to_labels > 1].index.tolist())
print("Test memiliki kolom label   :", "label" in test_df.columns)

folder_csv = train_df["filename"].str.split("/").str[1]
folder_disk = train_df["path"].apply(lambda p: os.path.basename(os.path.dirname(p)))
print("class_name tidak sesuai folder (CSV) :", (folder_csv != train_df["class_name"]).sum())
print("class_name tidak sesuai folder (disk):", (folder_disk != train_df["class_name"]).sum())

### 2.3 Distribusi jumlah dokumen per kelas
Jumlah dokumen dihitung untuk setiap kelas. Pada data training, proporsi setiap kelas berada pada kisaran 6%, sehingga ketidakseimbangan kelas dinilai tidak signifikan.

In [ ]:
class_dist = train_df.groupby(["label", "class_name"]).size().reset_index(name="jumlah")
class_dist["persen"] = (class_dist["jumlah"] / class_dist["jumlah"].sum() * 100).round(2)
display(class_dist)

n_total = len(train_df)
expected_uniform = n_total / len(class_dist)
print("Total gambar training:", f"{n_total:,}", "| target seimbang per kelas:", round(expected_uniform, 1))
print("Rasio max/min     :", round(class_dist["jumlah"].max() / class_dist["jumlah"].min(), 3))
print("Koefisien variasi :", round(class_dist["jumlah"].std() / class_dist["jumlah"].mean(), 4))

### 2.4 Visualisasi distribusi kelas dan imbalance
Bar chart memperlihatkan jumlah gambar pada setiap kelas dibandingkan garis seimbang sempurna. Distribusi yang relatif merata menunjukkan bahwa data tidak memerlukan pembobotan kelas (*class weighting*).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=class_dist, x="jumlah", y="class_name", color="steelblue", ax=ax)
ax.axvline(expected_uniform, color="red", linestyle="--", label=f"Seimbang sempurna ({expected_uniform:,.0f})")
for i, v in enumerate(class_dist["jumlah"]):
    ax.text(v, i, f" {v:,}", va="center")
ax.set_title(f"Jumlah gambar training per kelas (n = {n_total:,})")
ax.set_xlabel("Jumlah gambar")
ax.set_ylabel("Kelas")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 2.5 Membaca metadata gambar TIFF
Setiap file dibuka dengan PIL tanpa memuat seluruh piksel. Metadata yang dikumpulkan adalah lebar, tinggi, mode warna, format, jumlah frame, dan ukuran berkas. Mode `L` berarti grayscale dan `RGB` berarti tiga kanal warna. Kolom `n_frames` memastikan setiap berkas TIFF hanya berisi satu halaman.

In [ ]:
meta_train = read_image_metadata(train_df["path"].tolist())
meta_test = read_image_metadata(test_df["path"].tolist())
train_df = pd.concat([train_df, meta_train], axis=1)
test_df = pd.concat([test_df, meta_test], axis=1)

### 2.6 Mendeteksi gambar yang tidak terbaca
File yang gagal dibuka PIL diberi `format = ERROR`. Baris tersebut dihapus dari `train_df` sebelum tahap berikutnya, supaya DataLoader tidak berhenti saat membaca file rusak.

In [ ]:
print("Gagal dibaca train:", (train_df["format"] == "ERROR").sum())
print("Gagal dibaca test :", (test_df["format"] == "ERROR").sum())
display(train_df.loc[train_df["format"] == "ERROR", ["id", "filename"]])

In [ ]:
train_df = remove_unreadable(train_df)
print("Jumlah train setelah file rusak dibuang:", len(train_df))

### 2.7 Format, jumlah frame, dan mode warna
Seluruh citra harus berformat TIFF dengan satu frame. Seluruh citra pada data training dan data test berada pada mode `L` (skala keabuan). Karena itu, citra dibaca sebagai satu kanal lalu dikonversi menjadi tiga kanal (RGB) sebagai syarat masukan jaringan berbasis ResNet.

In [ ]:
print("Format train:", train_df["format"].value_counts().to_dict())
print("Format test :", test_df["format"].value_counts().to_dict())
print("Frame per file train:", train_df["n_frames"].value_counts().to_dict())
print("Frame per file test :", test_df["n_frames"].value_counts().to_dict())
print("Mode train:", train_df["mode"].value_counts().to_dict())
print("Mode test :", test_df["mode"].value_counts().to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="mode", data=train_df, color="steelblue", ax=ax)
ax.set_title("Mode Warna Gambar Training")
ax.set_xlabel("Mode")
ax.set_ylabel("Frekuensi")
plt.tight_layout()
plt.show()

### 2.8 Dimensi dan resolusi citra
Tinggi seluruh citra adalah 1.000 piksel, sedangkan distribusi lebar dan ukuran berkas pada data training dan data test relatif serupa. Data test diasumsikan berasal dari distribusi yang sama dengan data training.

In [ ]:
print(train_df[["width", "height", "size_kb"]].describe().round(2))
print(test_df[["width", "height", "size_kb"]].describe().round(2))

In [ ]:
ok_all = pd.concat([train_df.assign(split="train"), test_df.assign(split="test")], ignore_index=True)
ok_all["resolution"] = (ok_all["width"].astype(int).astype(str) + "x"
                        + ok_all["height"].astype(int).astype(str))

for s in ["train", "test"]:
    sub = ok_all[ok_all["split"] == s]
    top = sub["resolution"].value_counts()
    print(f"{s}: {top.size:,} resolusi unik")
    display(pd.DataFrame({"jumlah": top.head(5), "persen": (top.head(5) / len(sub) * 100).round(2)}))

In [ ]:
short_side = ok_all[["width", "height"]].min(axis=1)
aspect = ok_all["width"] / ok_all["height"]
print("Median sisi terpendek               :", short_side.median())
print("% sisi terpendek > 224 (diperkecil) :", round((short_side > 224).mean() * 100, 2))
print("% sisi terpendek < 224 (diperbesar) :", round((short_side < 224).mean() * 100, 2))
print("% aspect < 0.67 atau > 1.5          :", round(((aspect < 0.67) | (aspect > 1.5)).mean() * 100, 2))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=ok_all, x="width", hue="split", bins=60, element="step", stat="density",
             common_norm=False, ax=ax)
ax.set_title("Distribusi Lebar Gambar (piksel)")
ax.set_xlabel("Lebar (piksel)")
ax.set_ylabel("Densitas")
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=ok_all, x="size_kb", hue="split", bins=60, element="step", stat="density",
             common_norm=False, ax=ax)
ax.set_title("Distribusi Ukuran File (KB)")
ax.set_xlabel("Ukuran file (KB)")
ax.set_ylabel("Densitas")
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

### 2.9 Contoh gambar per kelas
Menampilkan satu contoh gambar untuk satu kelas. Ubah `LABEL_TO_SHOW` (0 sampai 15) untuk melihat kelas lain, dan `SAMPLE_NO` untuk contoh lain dari kelas yang sama. Satu plot dibuat per sel agar mudah dibandingkan.

In [ ]:
LABEL_TO_SHOW = 0
SAMPLE_NO = 0

row = (train_df[train_df["label"] == LABEL_TO_SHOW]
       .sample(frac=1, random_state=42).iloc[SAMPLE_NO])
with Image.open(row["path"]) as img:
    contoh = img.convert("RGB")

plt.figure(figsize=(5, 6))
plt.imshow(contoh)
plt.title(f"Label {LABEL_TO_SHOW}: {row['class_name']} | {row['width']:.0f}x{row['height']:.0f}")
plt.axis("off")
plt.show()

### 2.10 Deteksi duplikasi isi gambar
Duplikasi isi gambar dideteksi dengan nilai MD5 yang dihitung dari seluruh byte berkas. Pemeriksaan ini diperlukan karena:
* Duplikasi antara data latihan dan data validasi menyebabkan estimasi kinerja yang bias ke atas.
* Duplikasi dengan label yang berbeda mengindikasikan adanya kesalahan anotasi.

In [ ]:
train_df = add_md5(train_df)
test_df = add_md5(test_df)

In [ ]:
dup_mask = train_df["md5"].duplicated(keep=False)
print("Grup duplikat train                :", (train_df["md5"].value_counts() > 1).sum())
print("Baris train dalam grup duplikat    :", dup_mask.sum())
print("Duplikat di dalam test             :", test_df["md5"].duplicated().sum())
print("Baris test identik dengan train    :", test_df["md5"].isin(train_df["md5"]).sum())

label_per_group = train_df[dup_mask].groupby("md5")["label"].nunique()
print("Grup duplikat dengan label berbeda :", (label_per_group > 1).sum())
display(train_df.loc[dup_mask, "md5"].value_counts().head(10))

In [ ]:
dup_pct = (train_df[dup_mask].groupby("class_name").size()
           / train_df.groupby("class_name").size() * 100).fillna(0).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=dup_pct.values, y=dup_pct.index, color="steelblue", ax=ax)
ax.set_title("Persentase Gambar Training dalam Grup Duplikat per Kelas")
ax.set_xlabel("Persentase dari total gambar kelas (%)")
ax.set_ylabel("Kelas")
ax.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## Tahap 3: Preprocessing
Mengubah data mentah menjadi input yang siap dipakai model. Keputusan utama di tahap ini didasarkan pada hasil EDA: split yang mencegah kebocoran dari duplikat, resolusi yang mempertahankan rasio dokumen, augmentasi yang tidak merusak ciri dokumen, dan cache agar pembacaan TIFF tidak menjadi bottleneck.

### 3.1 Data cleaning
Pembersihan data dilakukan berdasarkan hasil pemeriksaan pada bagian 2.10, dengan fungsi `clean_duplicates` di `src/data.py`.
* Kelompok gambar identik dengan label yang berbeda dibuang seluruhnya, karena label yang benar tidak dapat ditentukan.
* Kelompok gambar identik dengan label yang sama dipertahankan satu salinan.

In [ ]:
before = len(train_df)
train_df = clean_duplicates(train_df)
print("Train sebelum:", before, "| sesudah:", len(train_df), "| dibuang:", before - len(train_df))

CLASS_NAMES = get_class_names(train_df)
NUM_CLASSES = len(CLASS_NAMES)
assert sorted(train_df["label"].unique()) == list(range(NUM_CLASSES)), "Label harus 0-15"
print(NUM_CLASSES, "kelas:", CLASS_NAMES)

### 3.2 Konfigurasi dan split train-validasi
Semua hyperparameter dikumpulkan dalam satu dict `CONFIG` di `src/config.py`. Pembagian data dilakukan dengan `StratifiedGroupKFold` sebanyak 5 fold, lalu satu fold dipakai sebagai validasi sehingga menghasilkan sekitar 80% training dan 20% validasi. Citra dengan md5 yang sama dikelompokkan sehingga tidak terbagi ke kedua subset.

In [ ]:
for key, value in CONFIG.items():
    print(f"{key:<16}: {value}")

In [ ]:
train_part, val_part = split_train_val(train_df, n_splits=CONFIG["n_splits"],
                                       random_state=CONFIG["random_state"])
print("Train:", len(train_part), "| Validasi:", len(val_part))
display(pd.DataFrame({
    "train_%": train_part["label"].value_counts(normalize=True).sort_index() * 100,
    "val_%": val_part["label"].value_counts(normalize=True).sort_index() * 100,
}).round(2))

In [ ]:
print("Citra validasi yang identik dengan training :", val_part["md5"].isin(train_part["md5"]).sum())
selisih = (train_part["label"].value_counts(normalize=True)
           - val_part["label"].value_counts(normalize=True)).abs()
print("Selisih proporsi kelas terbesar (%)        :", round(selisih.max() * 100, 2))
print("Gambar uji yang identik dengan data training:", test_df["md5"].isin(train_df["md5"]).sum())

### 3.3 Pre-resize dan cache gambar
Citra TIFF berukuran besar di-resize satu kali pada awal eksekusi, lalu hasilnya disimpan sebagai larik `uint8` dengan ukuran tetap. Pendekatan ini menghilangkan beban dekode berulang pada setiap epoch. Ruang disk diperiksa sebelum cache dibangun. Jika ruang tidak cukup, gambar dibaca langsung dari TIFF.

In [ ]:
TRAIN_CACHE, TEST_CACHE = None, None
if CONFIG["use_cache"]:
    TRAIN_CACHE, TEST_CACHE = build_all_caches(train_df, test_df, CONFIG["img_h"], CONFIG["img_w"], CACHE_DIR)
print("Cache train:", None if TRAIN_CACHE is None else TRAIN_CACHE.shape)

### 3.4 Transformasi
Augmentasi untuk training berupa rotasi maksimal 3 derajat, translasi maksimal 3%, skala 95% sampai 105%, dan jitter kecerahan serta kontras 10%.

Tidak ada flip horizontal maupun vertikal karena orientasi teks dan tata letak merupakan ciri kelas seperti letter, memo, dan form. Tidak ada random crop agresif karena bisa memotong bagian dokumen yang menjadi pembeda. Validasi dan test menggunakan transformasi deterministik tanpa augmentasi acak.

Normalisasi memakai mean dan std ImageNet karena bobot ResNet18 pretrained dilatih dengan normalisasi tersebut.

In [ ]:
train_tf, eval_tf = build_transforms(CONFIG["img_h"], CONFIG["img_w"], CONFIG["mean"], CONFIG["std"])
print("Transformasi training :", [type(t).__name__ for t in train_tf.transforms])
print("Transformasi evaluasi :", [type(t).__name__ for t in eval_tf.transforms])

### 3.5 Dataset dan DataLoader
Terdapat dua kelas Dataset di `src/data.py`. `DocumentTrainDataset` mengembalikan pasangan (gambar, label) untuk training dan validasi. `DocumentTestDataset` mengembalikan pasangan (gambar, id) karena data test tidak memiliki label dan id diperlukan untuk mencocokkan prediksi dengan `sample_submission.csv`.

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    train_part, val_part, test_df, train_tf, eval_tf,
    train_cache=TRAIN_CACHE, test_cache=TEST_CACHE,
    batch_size=CONFIG["batch_size"], num_workers=CONFIG["num_workers"],
)
val_ds = val_loader.dataset
print("Jumlah sampel | train:", len(train_loader.dataset), "| val:", len(val_ds),
      "| test:", len(test_loader.dataset))

### 3.6 Pemeriksaan pipeline
Memastikan bentuk batch, rentang label, rentang nilai setelah normalisasi, dan perilaku augmentasi sudah sesuai. Bentuk batch yang diharapkan adalah (32, 3, 384, 288), yaitu (batch, kanal, tinggi, lebar).

In [ ]:
x, y = next(iter(train_loader))
print("Shape batch train :", tuple(x.shape), "| label:", tuple(y.shape))
print("Rentang label     :", y.min().item(), "-", y.max().item())
print("Rentang nilai     :", round(x.min().item(), 3), "-", round(x.max().item(), 3))

In [ ]:
print("Augmentasi acak di train :", any(type(t).__name__ in ("RandomRotation", "RandomAffine")
                                        for t in train_tf.transforms))
print("Flip di train            :", any(type(t).__name__.endswith("Flip") for t in train_tf.transforms))
print("Transform validasi = eval:", val_ds.transform is eval_tf)
print("Transform test = eval    :", test_loader.dataset.transform is eval_tf)
print("Kolom label ada di test  :", "label" in test_df.columns)

### 3.7 Gambar sebelum dan sesudah preprocessing
Contoh satu citra validasi sebelum preprocessing. Setiap plot berada pada sel terpisah.

In [ ]:
i = 279
row = val_part.iloc[i]
with Image.open(row["path"]) as img:
    asli = img.convert("RGB")

plt.figure(figsize=(4, 5))
plt.imshow(asli)
plt.title(f"Asli | {row['mode']} {row['width']:.0f}x{row['height']:.0f} | {row['class_name']}")
plt.axis("off")
plt.show()

In [ ]:
# Gambar setelah preprocessing, dinormalisasi kembali agar bisa ditampilkan
mean_t = torch.tensor(CONFIG["mean"]).view(3, 1, 1)
std_t = torch.tensor(CONFIG["std"]).view(3, 1, 1)

x_i, y_i = val_ds[i]  # transformasi validasi bersifat deterministik
img_p = (x_i * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()

plt.figure(figsize=(4, 5))
plt.imshow(img_p)
plt.title(f"Setelah preprocessing | label {y_i}: {CLASS_NAMES[y_i]}")
plt.axis("off")
plt.show()

## Tahap 4: Modelling
Membangun model, loss, optimizer, scheduler, dan loop training. Target utamanya adalah validation accuracy yang stabil dan tinggi. Checkpoint dipilih berdasarkan validation loss terendah.

### 4.1 Arsitektur model
Arsitektur dipilih melalui `CONFIG["model_name"]` di `src/config.py`.
* `resnet18` (pilihan utama): ResNet18 dengan bobot pretrained ImageNet. Seluruh layer di-fine-tune dan lapisan fc diganti menjadi dropout lalu linear dengan 16 output. Definisinya ada di `src/model.py`.
* `docnet`: CNN buatan sendiri dengan empat blok konvolusi yang dilatih dari nol. Arsitektur ini juga didefinisikan di `src/model.py`.

In [ ]:
model = build_model(CONFIG, NUM_CLASSES, DEVICE)
print(f"Model: {CONFIG['model_name']} | parameter total: {sum(p.numel() for p in model.parameters()):,}")

model.eval()
with torch.no_grad():
    print("Shape output:", tuple(model(x.to(DEVICE)).shape))
model.train()

### 4.2 Inisialisasi loss, optimizer, dan scheduler
* Label smoothing 0,1 mencegah model terlalu yakin pada satu kelas, sehingga membantu menstabilkan validation accuracy.
* AdamW dengan learning rate 3e-4 dan weight decay 2e-2 untuk fine-tuning. Nilai weight decay diambil dari `CONFIG`.
* Cosine annealing menurunkan learning rate secara halus hingga mendekati nol pada epoch terakhir.
* Mixed precision (FP16) mempercepat training di GPU. GradScaler menjaga stabilitas gradien.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

### 4.3 Training dan validasi
Setiap epoch menjalankan satu pass training dan satu pass validasi. Checkpoint terbaik disimpan hanya ketika validation loss membaik. Early stopping menghentikan training jika validation loss tidak membaik selama `patience` epoch berturut-turut. Fungsi `train_model` di `src/engine.py` menjalankan seluruh proses ini.

In [ ]:
RESUME = False  # True untuk melanjutkan dari last checkpoint jika training terputus

result = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler,
                     DEVICE, CONFIG, CHECKPOINT_DIR, resume=RESUME)
history = result["history"]
best_epoch = result["best_epoch"]
BEST_PATH = result["best_path"]

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history["train_loss"], marker="o", label="Train loss")
plt.plot(epochs_ran, history["val_loss"], marker="o", label="Validation loss")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Checkpoint (epoch {best_epoch})")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross entropy loss")
plt.legend()
plt.grid(linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history["train_acc"], marker="o", label="Train accuracy")
plt.plot(epochs_ran, history["val_acc"], marker="o", label="Validation accuracy")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Checkpoint (epoch {best_epoch})")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## Tahap 5: Evaluation
### 5.1 Evaluasi model terbaik pada validasi
Checkpoint terbaik dimuat, lalu model dijalankan pada seluruh data validasi dalam mode evaluasi. Metrik yang digunakan adalah precision, recall, dan F1-score per kelas, beserta macro average dan weighted average.

In [ ]:
ckpt = load_checkpoint(model, BEST_PATH, DEVICE)
print("Checkpoint dimuat | epoch", ckpt["epoch"], "| val loss", round(ckpt["val_loss"], 4))

y_true, y_pred = predict_batches(model, val_loader, DEVICE)
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                            target_names=CLASS_NAMES, digits=3, zero_division=0))
print(f"Macro F1 validasi: {f1_score(y_true, y_pred, average='macro'):.4f}")

### 5.2 Confusion matrix
Memperlihatkan distribusi prediksi terhadap label sebenarnya. Elemen diagonal menunjukkan prediksi yang benar, sedangkan elemen di luar diagonal menunjukkan kesalahan. Pasangan kelas dengan frekuensi kesalahan tertinggi dianalisis untuk mengidentifikasi kelas yang memiliki kemiripan visual.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix Validasi")
plt.xlabel("Prediksi")
plt.ylabel("Label sebenarnya")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
off = cm.copy()
np.fill_diagonal(off, 0)
pairs = [(off[i, j], CLASS_NAMES[i], CLASS_NAMES[j])
         for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if off[i, j] > 0]
print("10 pasangan kesalahan terbesar (label sebenarnya -> prediksi):")
for n, actual, pred in sorted(pairs, reverse=True)[:10]:
    print(f"{actual:<24} -> {pred:<24} {n}")

### 5.3 Akurasi
Akurasi dihitung sebagai proporsi prediksi yang sesuai dengan label sebenarnya pada data validasi. Akurasi per kelas ditampilkan untuk melihat kelas yang paling lemah.

In [ ]:
y_true_np = np.array(y_true)
y_pred_np = np.array(y_pred)
correct = y_true_np == y_pred_np
print(f"Validation accuracy (checkpoint epoch {best_epoch}): {correct.mean():.4f}")
print(f"Benar: {correct.sum():,} dari {len(correct):,}")

per_class = pd.Series(correct).groupby(y_true_np).mean()
per_class.index = [CLASS_NAMES[i] for i in per_class.index]
print(per_class.sort_values().round(3))

## Tahap 6: Model Prediksi
Model dengan nilai validation loss terendah digunakan untuk memprediksi 7.994 citra uji. Hasilnya disimpan sebagai `outputs/submission.csv`, kemudian diverifikasi dari sisi urutan id, jumlah baris, rentang label, nama kolom, dan kelengkapan data.

In [ ]:
load_checkpoint(model, BEST_PATH, DEVICE)
submission = write_submission(model, test_loader, sample_sub, NUM_CLASSES, DEVICE,
                              os.path.join(OUTPUT_DIR, "submission.csv"))
display(submission.head())
print("Jumlah baris:", len(submission), "| missing:", submission.isna().sum().sum())